<a href="https://colab.research.google.com/github/drmuruga/Tardigrade-CAHS-d-simulation/blob/main/tardigrade_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tardigrade CAHS D Protein Gelation Pipeline
This notebook creates the project structure, builds the memory-safe simulation modules, and executes the OpenMM run.

In [3]:
# --- INSTALL REQUIRED LIBRARIES ---
!pip install -q py3Dmol requests

import requests
import py3Dmol

# --- STEP 1: FETCH SEQUENCE ---
print("1. Fetching CAHS D sequence (P0CU50) from UniProt...")
uniprot_url = "https://rest.uniprot.org/uniprotkb/P0CU50.fasta"
fasta_data = requests.get(uniprot_url).text

# Clean the FASTA format to extract purely the amino acid letters
sequence = "".join(fasta_data.split('\n')[1:]).strip()
print(f"   -> Successfully loaded {len(sequence)} amino acids.")

# --- STEP 2: GENERATE 3D STRUCTURE ---
print("\n2. Sending sequence to ESMFold API for 3D structure prediction...")
print("   (This usually takes 10-20 seconds. Please wait...)")
response = requests.post(
    "https://api.esmatlas.com/foldSequence/v1/pdb/",
    data=sequence
)

if response.status_code == 200:
    # Save the structure locally in Colab
    pdb_filename = "cahs_monomer.pdb"
    with open(pdb_filename, "w") as f:
        f.write(response.text)
    print(f"   -> Success! Structure seamlessly generated and saved as '{pdb_filename}'.")

    # --- STEP 3: VISUALIZE 3D STRUCTURE ---
    print("\n3. Rendering 3D model...")
    with open(pdb_filename, 'r') as f:
        pdb_data = f.read()

    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_data, 'pdb')

    # Apply the custom segment styling for your specific domains
    view.setStyle({'resi': '1-89'}, {'cartoon': {'color': 'blue'}})
    view.setStyle({'resi': '90-191'}, {'cartoon': {'color': 'green'}})
    view.setStyle({'resi': '192-227'}, {'cartoon': {'color': 'firebrick'}})

    view.zoomTo()
    view.show()
else:
    print(f"\n   -> Error: ESMFold API responded with status code {response.status_code}")
    print(response.text)

1. Fetching CAHS D sequence (P0CU50) from UniProt...
   -> Successfully loaded 227 amino acids.

2. Sending sequence to ESMFold API for 3D structure prediction...
   (This usually takes 10-20 seconds. Please wait...)
   -> Success! Structure seamlessly generated and saved as 'cahs_monomer.pdb'.

3. Rendering 3D model...


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [4]:
# --- PHASE 2: SYSTEM PREPARATION & HYDRATION ---
import sys
import subprocess

# 1. Install required libraries robustly
print("1. Ensuring pdbfixer is installed in the active environment...")
try:
    from pdbfixer import PDBFixer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pdbfixer"])
    from pdbfixer import PDBFixer

from openmm.app import ForceField, Modeller, PME
from openmm import unit

# 2. Repair Topology with PDBFixer
print("\n2. Loading and repairing 'cahs_monomer.pdb' topology...")
fixer = PDBFixer(filename='cahs_monomer.pdb')

# Add missing terminal caps and hydrogens (Crucial for ESMFold raw outputs)
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.0)
print("   -> Topology repaired. Terminal caps and hydrogens added at pH 7.0.")

# 3. Setup Force Field and Solvation
print("\n3. Setting up Amber14 force field and explicit solvent box...")
forcefield = ForceField('amber14-all.xml', 'amber14/tip3p.xml')
modeller = Modeller(fixer.topology, fixer.positions)

# Submerge protein in a water box with 0.8 nm padding and standard 0.15M ions
modeller.addSolvent(forcefield, padding=0.8 * unit.nanometer, ionicStrength=0.15 * unit.molar)

# 4. Build System
print("\n4. Building the final simulation system...")
# Using the correct PME import to avoid the previous AttributeError
system = forcefield.createSystem(modeller.topology, nonbondedMethod=PME, nonbondedCutoff=1.0 * unit.nanometer)

print("   -> Success! System built.")
print(f"   -> Total atoms in the simulation box: {modeller.topology.getNumAtoms()}")

1. Ensuring pdbfixer is installed in the active environment...

2. Loading and repairing 'cahs_monomer.pdb' topology...
   -> Topology repaired. Terminal caps and hydrogens added at pH 7.0.

3. Setting up Amber14 force field and explicit solvent box...

4. Building the final simulation system...
   -> Success! System built.
   -> Total atoms in the simulation box: 833331


In [2]:
# 1. Delete the restrictive version pin
!rm /usr/local/conda-meta/pinned

# 2. Install OpenMM with CUDA support
!mamba install -c conda-forge openmm -y


Looking for: ['openmm']

conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache

Pinned packages:
  - python 3.11.*


Transaction

  Prefix: /usr/local

  Updating specs:

   - openmm
   - ca-certificates
   - certifi
   - openssl


  Package               Version  Build                 Channel           Size
───────────────────────────────────────────────────────────────────────────────
  Install:
───────────────────────────────────────────────────────────────────────────────

  + cuda-version           13.3  hcbadf70_3            conda-forge       22kB
  + rocm-core             7.2.4  h54a6638_0            conda-forge       37kB
  + libgfortran5         14.2.0  hf1ad2bd_2            conda-forge        1MB
  + opencl-headers   2025.06.13  hecca717_0            conda-forge       56kB
  + libcufft          12.3.0.29  hecca717_0            conda-forge      151MB
  + cuda-nvrtc          13.3.33  hecc

In [ ]:
from openmm import LangevinIntegrator, Platform
from openmm.app import Simulation, StateDataReporter, PDBReporter
from openmm import unit
import sys

print("1. Initializing Simulation Environment...")
integrator = LangevinIntegrator(300 * unit.kelvin, 1 / unit.picosecond, 2.0 * unit.femtoseconds)

# Let's test the newly minted CUDA installation platform
platform = Platform.getPlatformByName('CUDA') if 'CUDA' in [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())] else Platform.getPlatformByName('CPU')
print(f"   -> Using compute platform: {platform.getName()}")

simulation = Simulation(modeller.topology, system, integrator, platform)
simulation.context.setPositions(modeller.positions)

print("\n2. Performing Energy Minimization...")
simulation.minimizeEnergy()
print("   -> Energy minimization complete!")

print("\n3. Running Production Simulation (50,000 steps)...")
simulation.reporters.append(StateDataReporter(sys.stdout, 2000, step=True, potentialEnergy=True, temperature=True, speed=True))
simulation.reporters.append(PDBReporter('cahs_trajectory.pdb', 5000))

simulation.step(50000)
print("\n   -> Simulation complete! Trajectory saved to 'cahs_trajectory.pdb'.")

1. Initializing Simulation Environment...
   -> Using compute platform: CPU

2. Performing Energy Minimization...
